# SK Scout — Shravani's Exploration Notebook
### GitHub Anomaly & Bot Detection
This notebook mirrors Kanak's `exploration.ipynb` but adds **4 new detection layers**:
1. 🔴 **Suspicious Human Accounts** – bot-like behaviour without [bot] tag
2. 🟠 **Lockstep Detection** – coordinated activity across ALL event types
3. 🟡 **Phishing Repo Names** – StarScout-inspired keyword matching
4. 🟣 **Branch Explosion + AI Co-author** – structural repo anomalies

In [1]:
import gzip, json, re
import pandas as pd
import numpy as np

# ── Step 1: Download one hour of GHArchive data ──────────────────────────────
import requests, os
from pathlib import Path

RAW_DIR = Path('data/raw/gharchive')
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Change date/hour as you like
DT = '2026-04-15-12'
FILE = RAW_DIR / f'{DT}.json.gz'

if not FILE.exists():
    url = f'https://data.gharchive.org/{DT}.json.gz'
    print(f'Downloading {url} ...')
    with requests.get(url, stream=True, timeout=90) as r:
        r.raise_for_status()
        with open(FILE, 'wb') as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)
    print(f'Saved: {FILE.stat().st_size/1e6:.1f} MB')
else:
    print(f'Using cached: {FILE}')

Using cached: data/raw/gharchive/2026-04-15-12.json.gz


In [2]:
# ── Step 2: Parse events (same as Kanak + new fields) ────────────────────────

KNOWN_BOT_RE = re.compile(
    r'\[bot\]|-bot$|^bot-|dependabot|renovate|github-actions|codecov|snyk-bot',
    re.IGNORECASE
)
PHISH_RE = re.compile(
    r'crack|free|hack|cheat|stealer|wallet|crypto|autoclicker|solana|roblox|adobe|keygen',
    re.IGNORECASE
)
AI_RE = re.compile(r'claude|copilot|chatgpt|openai|gpt-?4|gemini', re.IGNORECASE)

rows = []
MAX_EVENTS = 5000  # increase for bigger sample

with gzip.open(FILE, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= MAX_EVENTS:
            break
        try:
            ev = json.loads(line)
        except:
            continue

        login     = ev.get('actor', {}).get('login')
        repo_name = ev.get('repo', {}).get('name', '')
        ev_type   = ev.get('type', '')

        # AI co-author check in commits
        ai_coauthor = False
        for c in ev.get('payload', {}).get('commits', []):
            if AI_RE.search(c.get('message','')) or AI_RE.search(c.get('author',{}).get('name','')):
                ai_coauthor = True
                break

        rows.append({
            'event_id':     ev.get('id'),
            'event_type':   ev_type,
            'actor_login':  login,
            'actor_id':     ev.get('actor', {}).get('id'),
            'repo_name':    repo_name,
            'repo_id':      ev.get('repo', {}).get('id'),
            'created_at':   ev.get('created_at'),
            'is_public':    ev.get('public'),
            'is_known_bot': bool(KNOWN_BOT_RE.search(login)) if login else False,
            'phish_name':   bool(PHISH_RE.search(repo_name.split('/')[-1])),
            'ai_coauthor':  ai_coauthor,
            'payload_size': len(line),
        })

df = pd.DataFrame(rows)
df['created_at'] = pd.to_datetime(df['created_at'], utc=True)
print(f'Parsed {len(df)} events')
df.head()

Parsed 5000 events


,event_id,event_type,actor_login,actor_id,repo_name,repo_id,created_at,is_public,is_known_bot,phish_name,ai_coauthor,payload_size
0,10610926681,PushEvent,bilibiliSilentH,167333048,bilibiliSilentH/Bili_Ticket_Monitor,1211438376,2026-04-15 12:00:00+00:00,True,False,False,False,641
1,10610926685,PushEvent,github-actions[bot],41898282,UnaScott/Scott,748181688,2026-04-15 12:00:00+00:00,True,True,False,False,604
2,10610926694,PushEvent,mauricegift,200888291,mauricegift/free-proxies,1072646216,2026-04-15 12:00:00+00:00,True,False,True,False,609
3,10610926710,DeleteEvent,github-actions[bot],41898282,vgijssel/hermit-python-packages,981976343,2026-04-15 12:00:00+00:00,True,True,False,False,735
4,10610926714,PushEvent,saarors,171342946,saarors/saarors,1184638091,2026-04-15 12:00:00+00:00,True,False,False,False,577


In [3]:
# ── Step 3: Basic overview (same as Kanak) ────────────────────────────────────
print('Event types:')
print(df['event_type'].value_counts())
print(f'\nUnique repos:  {df["repo_name"].nunique()}')
print(f'Unique actors: {df["actor_login"].nunique()}')
print(f'Known bots:    {df["is_known_bot"].sum()}')
print(f'Phish names:   {df["phish_name"].sum()}')
print(f'AI co-author:  {df["ai_coauthor"].sum()}')

Event types:
event_type
PushEvent                        4284
CreateEvent                       460
DeleteEvent                       190
PullRequestEvent                   18
IssuesEvent                        14
PullRequestReviewEvent             12
IssueCommentEvent                   8
PullRequestReviewCommentEvent       7
WatchEvent                          5
ForkEvent                           1
ReleaseEvent                        1
Name: count, dtype: int64

Unique repos:  3951
Unique actors: 3243
Known bots:    859
Phish names:   46
AI co-author:  0


In [5]:
# ── Step 4: Per-repo stats (same as Kanak + new signals) ─────────────────────
repo_stats = df.groupby('repo_name').agg(
    total_events    =('event_id',    'count'),
    unique_actors   =('actor_login', 'nunique'),
    bot_events      =('is_known_bot','sum'),
    phish_name_flag =('phish_name',  'max'),
    ai_coauthor_flag=('ai_coauthor', 'max'),
).reset_index()

repo_stats['events_per_actor'] = repo_stats['total_events'] / repo_stats['unique_actors'].replace(0,1)
repo_stats['bot_ratio']        = repo_stats['bot_events']   / repo_stats['total_events']
repo_stats['low_actor_flag']   = repo_stats['unique_actors'] <= 1
repo_stats['high_activity']    = repo_stats['total_events'] >= 5

# Composite suspicion score
repo_stats['suspicious_score'] = (
    (repo_stats['events_per_actor'] > 5 ).astype(int) +
    (repo_stats['bot_ratio']        > 0.5).astype(int) +
    (repo_stats['unique_actors']    <= 1 ).astype(int) +
    repo_stats['phish_name_flag'].astype(int) * 3 +   # NEW – high weight
    repo_stats['ai_coauthor_flag'].astype(int) * 2    # NEW
)

repo_stats.sort_values('suspicious_score', ascending=False).head(15)

,repo_name,total_events,unique_actors,bot_events,phish_name_flag,ai_coauthor_flag,events_per_actor,bot_ratio,low_actor_flag,high_activity,suspicious_score
848,Nam17111998/Udemy_free,1,1,1,True,False,1.0,1.0,True,False,5
2275,hhackk/hacknews,1,1,1,True,False,1.0,1.0,True,False,5
2328,iamfour07/crypto-scanner,1,1,1,True,False,1.0,1.0,True,False,5
1319,YaChengMu/Free-Node,1,1,1,True,False,1.0,1.0,True,False,5
2762,mauricegift/free-proxies,11,1,0,True,False,11.0,0.0,True,True,5
2255,hayaseta/solana-sniping-bot,1,1,1,True,False,1.0,1.0,True,False,5
2245,haoyousun60-create/TrendRadar-Crypto,1,1,1,True,False,1.0,1.0,True,False,5
3141,pranjalmandhan/crypto-market-pipeline,1,1,1,True,False,1.0,1.0,True,False,5
1211,Theikk/freemagic_rss_feeds,1,1,0,True,False,1.0,0.0,True,False,4
3886,yeong-hwan/buidlhack-2026-bnb,1,1,0,True,False,1.0,0.0,True,False,4


In [6]:
# ── Step 5: 🔴 NEW – Suspicious Human Accounts ───────────────────────────────
from scipy.stats import entropy as sp_entropy

actor_records = []
for login, grp in df[~df['is_known_bot']].groupby('actor_login'):
    grp = grp.sort_values('created_at')
    type_counts = grp['event_type'].value_counts(normalize=True)
    ev_entropy  = float(sp_entropy(type_counts))

    times = grp['created_at'].dropna().sort_values()
    gaps  = times.diff().dt.total_seconds().dropna()
    burst_frac = float((gaps < 60).sum() / len(gaps)) if len(gaps) else 0.0

    actor_records.append({
        'actor_login':      login,
        'total_events':     len(grp),
        'unique_repos':     grp['repo_name'].nunique(),
        'unique_types':     grp['event_type'].nunique(),
        'event_entropy':    ev_entropy,
        'burst_fraction':   burst_frac,
        'ai_coauthor':      bool(grp['ai_coauthor'].any()),
    })

actors_df = pd.DataFrame(actor_records)
actors_df['susp_score'] = (
    (actors_df['event_entropy']  < 0.5).astype(int) +
    (actors_df['burst_fraction'] > 0.6).astype(int) +
    (actors_df['total_events']   > 20 ).astype(int) +
    (actors_df['unique_types']   == 1 ).astype(int) +
    actors_df['ai_coauthor'].astype(int)
)

print('Top suspicious human accounts:')
actors_df.sort_values('susp_score', ascending=False).head(15)

Top suspicious human accounts:


,actor_login,total_events,unique_repos,unique_types,event_entropy,burst_fraction,ai_coauthor,susp_score
2774,sidarthus89,51,2,1,0.0,1.000000,False,4
501,KarolisZemaitis,21,1,1,0.0,1.000000,False,4
871,SoliSpirit,39,2,1,0.0,1.000000,False,4
570,MIchael-wufan,30,1,1,0.0,1.000000,False,4
2069,lakshit77,30,1,1,0.0,1.000000,False,4
2165,maldwg,4,1,1,0.0,1.000000,False,3
2552,radu-lzr,2,1,1,0.0,1.000000,False,3
1062,adem4321,6,1,1,0.0,1.000000,False,3
2545,quocloly83-cpu,2,1,1,0.0,1.000000,False,3
2162,maitrungduc1410,2,1,1,0.0,1.000000,False,3


In [7]:
# ── Step 6: 🟠 NEW – Lockstep Detection ──────────────────────────────────────
df2 = df.copy()
df2['window'] = df2['created_at'].dt.floor('30min')

lockstep = (
    df2.groupby(['repo_name','window'])
    .agg(actor_count=('actor_login','nunique'),
         event_count=('event_id','count'),
         actors=('actor_login', lambda x: ','.join(sorted(x.dropna().unique()))))
    .reset_index()
)

lockstep_hits = lockstep[lockstep['actor_count'] >= 3].sort_values('actor_count', ascending=False)
print(f'Lockstep windows (3+ actors): {len(lockstep_hits)}')
lockstep_hits.head(15)

Lockstep windows (3+ actors): 1


,repo_name,window,actor_count,event_count,actors
950,PostHog/posthog,2026-04-15 12:00:00+00:00,5,11,"VojtechBartos,a-lider,graphite-app[bot],paulda..."


In [8]:
# ── Step 7: 🟡 NEW – Phishing Repo Names ─────────────────────────────────────
phish_repos = repo_stats[repo_stats['phish_name_flag']].sort_values('suspicious_score', ascending=False)
print(f'Repos with phishing keywords in name: {len(phish_repos)}')
phish_repos.head(15)

Repos with phishing keywords in name: 33


,repo_name,total_events,unique_actors,bot_events,phish_name_flag,ai_coauthor_flag,events_per_actor,bot_ratio,low_actor_flag,high_activity,suspicious_score
3141,pranjalmandhan/crypto-market-pipeline,1,1,1,True,False,1.0,1.0,True,False,5
1319,YaChengMu/Free-Node,1,1,1,True,False,1.0,1.0,True,False,5
2328,iamfour07/crypto-scanner,1,1,1,True,False,1.0,1.0,True,False,5
2275,hhackk/hacknews,1,1,1,True,False,1.0,1.0,True,False,5
2255,hayaseta/solana-sniping-bot,1,1,1,True,False,1.0,1.0,True,False,5
2245,haoyousun60-create/TrendRadar-Crypto,1,1,1,True,False,1.0,1.0,True,False,5
2762,mauricegift/free-proxies,11,1,0,True,False,11.0,0.0,True,True,5
848,Nam17111998/Udemy_free,1,1,1,True,False,1.0,1.0,True,False,5
3767,vmheaven/VMHeaven-Free-Proxy-Updated,1,1,0,True,False,1.0,0.0,True,False,4
3439,simear2004/foo_openhacks_mod,1,1,0,True,False,1.0,0.0,True,False,4


In [9]:
# ── Step 8: 🟣 NEW – AI Co-author Repos ──────────────────────────────────────
ai_repos = repo_stats[repo_stats['ai_coauthor_flag']].sort_values('suspicious_score', ascending=False)
print(f'Repos with AI handle in commit authors: {len(ai_repos)}')
ai_repos

Repos with AI handle in commit authors: 0


,repo_name,total_events,unique_actors,bot_events,phish_name_flag,ai_coauthor_flag,events_per_actor,bot_ratio,low_actor_flag,high_activity,suspicious_score


In [10]:
# ── Step 9: Summary dashboard ─────────────────────────────────────────────────
print('=== SK Scout Summary ===')
print(f'Total events parsed:              {len(df):,}')
print(f'Unique repos:                     {df["repo_name"].nunique():,}')
print(f'Unique actors:                    {df["actor_login"].nunique():,}')
print(f'Known bot events:                 {df["is_known_bot"].sum():,}')
print(f'Phishing-keyword repos:           {df["phish_name"].sum():,}')
print(f'AI co-author events:              {df["ai_coauthor"].sum():,}')
print(f'Suspicious human accounts (≥2):  {(actors_df["susp_score"] >= 2).sum():,}')
print(f'Lockstep windows (3+ actors):    {len(lockstep_hits):,}')
print(f'High-risk repos (score ≥ 3):     {(repo_stats["suspicious_score"] >= 3).sum():,}')

=== SK Scout Summary ===
Total events parsed:              5,000
Unique repos:                     3,951
Unique actors:                    3,243
Known bot events:                 859
Phishing-keyword repos:           46
AI co-author events:              0
Suspicious human accounts (≥2):  3,113
Lockstep windows (3+ actors):    1
High-risk repos (score ≥ 3):     35
